# RR BENCHMARK PHASE B3: tie audit + the published mesenchymal-only (ASC) secondary
Same frozen inputs as B2 (signatures_frozen_B1b.json 1457464f...54e9). Cannot change the primary result.
(1) Exact tie-block midrank per row, and which drug IDs sit on it. (2) The secondary declared in the published scoping paper, Section 9: scoring restricted to ASC-line signatures. Reported, never gating.
Run: hamburger > Runtime > Run all. Paste back everything between `=== PHASE B3 REPORT BEGIN ===` and `=== PHASE B3 REPORT END ===`.

In [ ]:
# CELL 1: load frozen inputs, recover each library signature's cell line
from google.colab import drive
drive.mount("/content/drive")
import os,json,hashlib,time,numpy as np,pandas as pd,subprocess,sys
W="/content/drive/MyDrive/RitschelResearch/benchmark"; os.chdir(W)
subprocess.run([sys.executable,"-m","pip","install","-q","fsspec","aiohttp","h5py"],check=False)
import fsspec,h5py
raw=open("signatures_frozen_B1b.json","rb").read(); assert hashlib.sha256(raw).hexdigest()=="1457464f13bac9a7bdeca438f44414f4a2c2a7e0562992375b982217fd5c54e9"
FZ=json.loads(raw); SIG=FZ["signatures"]; DR=pd.DataFrame(FZ["drugs"])
ROWS=["6","8","10","12","14/15","17","18"]; DRUGROW={"6":["6"],"8":["8"],"10":["10"],"12":["12"],"14/15":["14"],"17":["17"],"18":["18"]}
M=np.load("libA_matrix.npy"); perts=np.load("libA_perts.npy",allow_pickle=True); genes=np.load("libA_genes.npy"); GI={int(x):i for i,x in enumerate(genes)}
# Rebuild the library row order exactly as Phase A built it (sorted by gctx column position), now keeping cell line
s=pd.read_csv("siginfo_beta.txt",sep="\t",usecols=["sig_id","pert_id","pert_type","is_exemplar_sig","qc_pass","cell_iname"],low_memory=False)
s=s[(s.pert_type=="trt_cp")&(s.is_exemplar_sig==1)&(s.qc_pass==1)]
u="https://s3.amazonaws.com/macchiato.clue.io/builds/LINCS2020/level5/level5_beta_trt_cp_n720216x12328.gctx"
h=h5py.File(fsspec.open(u,"rb").open(),"r"); colid=np.array([x.decode() for x in h["0"]["META"]["COL"]["id"][:]])
pos={x:i for i,x in enumerate(colid)}
want=sorted((pos[x],p,c) for x,p,c in zip(s.sig_id,s.pert_id,s.cell_iname) if x in pos)
cells=np.array([w[2] for w in want]); p2=np.array([w[1] for w in want])
assert len(p2)==len(perts) and (p2==perts).all(), "LIBRARY ORDER NOT REPRODUCED"
print("library order reproduced;",len(cells),"signatures; ASC signatures:",int((cells=="ASC").sum()))

In [ ]:
# CELL 2: B2 scoring code, unchanged
def ks(rp,n,t):
    if t==0 or n==0: return 0.0
    V=np.sort(np.asarray(rp,float)); j=np.arange(1,t+1,dtype=float)
    a=np.max(j/t-V/n); b=np.max(V/n-(j-1.0)/t)
    return float(a) if a>b else float(-b)
def conn_all(Mx,up,dn):
    n=Mx.shape[1]; out=np.empty(Mx.shape[0])
    order=np.argsort(-Mx,axis=1,kind="stable"); rank=np.empty_like(order); ar=np.arange(1,n+1)
    np.put_along_axis(rank,order,np.broadcast_to(ar,order.shape),axis=1)
    for i in range(Mx.shape[0]):
        eu=ks(rank[i,up],n,len(up)); ed=ks(rank[i,dn],n,len(dn)); out[i]=0.0 if eu*ed>=0 else (eu-ed)/2.0
    return out
def midrank(v):
    v=np.asarray(v,float); n=v.size; o=np.argsort(v,kind="stable"); r=np.empty(n); i=0
    while i<n:
        j=i
        while j+1<n and v[o[j+1]]==v[o[i]]: j+=1
        r[o[i:j+1]]=0.5*(i+j)+1.0; i=j+1
    return r
def score(rowmask):
    PCT={};TIEMID={};COMP={}
    for row in ROWS:
        S=SIG[row]; cols=[GI[e] for e in S["usable"]]; p={e:i for i,e in enumerate(S["usable"])}
        up=np.array([p[e] for e in S["up"]]); dn=np.array([p[e] for e in S["dn"]])
        sc=conn_all(M[rowmask][:,cols],up,dn)
        comp=pd.DataFrame({"p":perts[rowmask],"s":sc}).groupby("p")["s"].median()
        pct=pd.Series((midrank(comp.values)-0.5)/len(comp),index=comp.index)
        z=comp==0; PCT[row]=pct; COMP[row]=comp; TIEMID[row]=float(pct[z].iloc[0]) if z.any() else None
    return PCT,TIEMID,COMP
def summarize(PCT,label):
    out={}
    for row in ROWS:
        drugs=DR[DR.row.astype(str).isin(DRUGROW[row])]; per=[]
        for d,g in drugs.groupby("drug"):
            v=[PCT[row][p] for p in g.pert_id if p in PCT[row].index]
            per.append(float(np.median(v)) if v else None)
        per=[x for x in per if x is not None]
        out[row]=float(np.median(per)) if per else None
    return out

In [ ]:
# CELL 3: tie audit on the primary (all lines). Must reproduce B2 exactly.
t0=time.time(); PCT,TIEMID,COMP=score(np.ones(len(perts),bool))
B2={"6":0.6499,"8":0.2992,"10":0.5041,"12":0.1764,"14/15":0.5364,"17":0.1542,"18":0.3887}
prim=summarize(PCT,"all")
for r in ROWS: assert abs(prim[r]-B2[r])<6e-5, ("DOES NOT REPRODUCE B2",r,prim[r],B2[r])
TIES=[]
for row in ROWS:
    drugs=DR[DR.row.astype(str).isin(DRUGROW[row])]
    for d,g in drugs.groupby("drug"):
        for p in g.pert_id: TIES.append((row,d,p,float(COMP[row][p]),float(PCT[row][p]),COMP[row][p]==0))
print("B2 reproduced;",round(time.time()-t0),"s")

In [ ]:
# CELL 4: ASC-only secondary (declared in the published scoping paper). Not gating. Same permutation null, 10,000 draws.
asc=cells=="ASC"
PA,TA,CA=score(asc)
sec=summarize(PA,"ASC")
scored=[r for r in ROWS if sec[r] is not None]
STAT=float(np.mean([sec[r] for r in scored])) if scored else None
rng=np.random.default_rng(20261001); NULL=[]
STRUCT={row:[(d,sum(p in PA[row].index for p in g.pert_id)) for d,g in DR[DR.row.astype(str).isin(DRUGROW[row])].groupby("drug")] for row in scored}
for b in range(10000):
    tot=0
    for row in scored:
        arr=PA[row].values; need=sum(k for d,k in STRUCT[row] if k>0); pick=rng.choice(len(arr),size=need,replace=False); i=0; per=[]
        for d,k in STRUCT[row]:
            if k==0: continue
            per.append(np.median(arr[pick[i:i+k]])); i+=k
        tot+=np.median(per)
    NULL.append(tot/len(scored))
NULL=np.array(NULL); pA=(1+int((NULL<=STAT).sum()))/(len(NULL)+1) if scored else None

In [ ]:
# CELL 5: report
print("=== PHASE B3 REPORT BEGIN ===")
print("primary reproduced B2 on all 7 rows")
print("TIE AUDIT (row, drug, pert_id, compound score, percentile, ON TIE BLOCK):")
for row,d,p,sc,pc,z in TIES: print(f"  {row:6s} {d:15s} {p} score {sc:+.5f} pct {pc:.4f} {'TIE' if z else ''}")
for r in ROWS: print(f"  row {r}: tie-block midrank {TIEMID[r]:.4f}  library exact-zero {float((COMP[r]==0).mean()):.3f}")
print("ASC-ONLY SECONDARY:", int(asc.sum()),"ASC signatures;", {r:len(CA[r]) for r in ROWS} if CA else "")
for r in ROWS: print(f"  row {r}: {'drug has no ASC signature' if sec[r] is None else f'{sec[r]:.4f}'}")
print("  rows scored",scored,"mean",None if STAT is None else round(STAT,4),"null mean",round(float(NULL.mean()),4),"sd",round(float(NULL.std()),4),"p",pA)
print("=== PHASE B3 REPORT END ===")